In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

In [2]:
class MyState(TypedDict):
    message:str
    result:str

In [22]:
def greet_node(state: MyState) -> dict:
    return {"result":f'Hello sir how are {state['message']}'}

def shout_node(state:MyState) -> dict:
    print(state)
    return {'result': f"Shout how are you {state['result']}"}

In [23]:
graph=StateGraph(MyState)
graph.add_node("greet",greet_node)
graph.add_node("shout",shout_node)
graph.set_entry_point("greet")
graph.add_edge("greet","shout")
graph.add_edge("shout",END)

In [24]:
app=graph.compile()
result=app.invoke({"message":"Greetings!!","result":''})
print(result)

{'message': 'Greetings!!', 'result': 'Hello sir how are Greetings!!'}
{'message': 'Greetings!!', 'result': 'Shout how are you Hello sir how are Greetings!!'}


### Reducers

In [25]:
from typing import TypedDict,Annotated
from langgraph.graph import StateGraph,END
import operator

class ConversationState(TypedDict):
    messages:Annotated[list,operator.add]
    message_count:int


def node_a(state:ConversationState):
    return {
        "messages":["Hey in node a"],
        "message_count":state['message_count']+1
    }

def node_b(state:ConversationState):
    return {
        "messages":["Hey in node b"],
        "message_count":state['message_count']+1
    }

graph=StateGraph(ConversationState)
graph.add_node('a',node_a)
graph.add_node('b',node_b)
graph.add_edge('a','b')
graph.add_edge('b',END)
graph.set_entry_point('a')

In [26]:
app=graph.compile()
result=app.invoke({"messages":[],"message_count":0})
result

{'messages': ['Hey in node a', 'Hey in node b'], 'message_count': 2}

### Excercise

In [32]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator


class WordState(TypedDict):
    input_str: str
    words:Annotated[list,operator.add]
    word_count:int
    report:str


def parse(state: WordState) -> dict:
    return {"words":state['input_str'].split()}

# TODO: Node 2 - count the words
def count(state: WordState) -> dict:
    return {"word_count":len(state['words'])}

# TODO: Node 3 - format a report
def report(state: WordState) -> dict:
    unique_words=str(set(state['words']))
    longest_word=''
    max_len=0
    for word in state['words']:
        if len(word)>max_len:
            longest_word=word
            max_len=len(word)

    return {"report":f"Report for input sentence {state['input_str']}\n \
            length of sentence {state['word_count']}\n \
            unique words: {unique_words}\n \
            longest word:{longest_word}"}


graph=StateGraph(WordState)
graph.add_node("parse",parse)
graph.add_node("count",count)
graph.add_node("report",report)
graph.add_edge("parse","count")
graph.add_edge("count","report")
graph.add_edge("report",END)

graph.set_entry_point("parse")


test_sentence = "The quick brown fox jumps over the lazy dog"
app=graph.compile()
result=app.invoke({'input_str':test_sentence,'words':[],'word_count':0,'report':''})
print(result)

{'input_str': 'The quick brown fox jumps over the lazy dog', 'words': ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog'], 'word_count': 9, 'report': "Report for input sentence The quick brown fox jumps over the lazy dog\n             length of sentence 9\n             unique words: {'the', 'dog', 'fox', 'quick', 'lazy', 'over', 'brown', 'The', 'jumps'}\n             longest word:quick"}


### Sequential Langgraph

In [37]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()

True

In [38]:
llm=ChatOpenAI(model='gpt-4.1-mini',temperature=0.001)
class PipelineState(TypedDict):
    topic:str
    outline:str
    draft:str
    final_article:str

In [39]:
def create_outline(states:PipelineState)->dict:
    response=llm.invoke([HumanMessage(content=f"Create a 5 point article about: {states['topic']} and generate an outline for the topic covering various aspects for the topic")])
    return {'outline':response.content}

def create_draft(states:PipelineState)->dict:
    response=llm.invoke([HumanMessage(content=f"Using the outline Write me a draft for the topic {states['topic']} Outline for the topic is given as: {states['outline']}")])
    return {'draft':response.content}

def final_article(states:PipelineState)->dict:
    response=llm.invoke([HumanMessage(content=f"Using the draft provided to you write me a 300 worded article about the topic {states['topic']} The draft is given below as: {states['draft']} ")])
    return {'final_article':response.content}


In [41]:
# task flow topic --> outline --> draft -->final flow
graph=StateGraph(PipelineState)
graph.add_node("outline",create_outline)
graph.add_node("draft",create_draft)
graph.add_node("final_article",final_article)

graph.add_edge("outline","draft")
graph.add_edge("draft",'final_article')
graph.add_edge("final_article",END)

graph.set_entry_point("outline")

app=graph.compile()

In [44]:
result = app.invoke({
    "topic":"Why Python is great for AI development",
    "outline":"",
    "draft": "",
    "final_article": ""
})

APIConnectionError: Connection error.